# Chatbot with Message History

In this project, I am building and implementing an LLM-powered chatbot that can have a conversation and remember previous interactions.

The chatbot uses a language model to generate responses while maintaining conversation history, allowing it to understand and respond based on previous messages.

This project covers the fundamentals of building a conversational chatbot and provides a foundation for more advanced applications such as:

Conversational RAG: Building a chatbot that can have conversations over external sources of data.
Agents: Building chatbots that can take actions and interact with external tools.

The concepts covered in this project provide a foundation for understanding and developing these more advanced LLM applications.

## Basic Chatbot with LLM

### Setting Up Environment Variables

In [87]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

### Initializing the Chat Model

In [88]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [89]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020ECB11BDC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020EDB0E3DC0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Sending a Single Message

In [90]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Dev and I am a Software Engineer")])

AIMessage(content='Hello Dev! 👋 Great to meet a fellow software engineer. How can I help you today? Whether it’s a coding question, architecture discussion, debugging assistance, or anything else tech‑related, just let me know!', additional_kwargs={'reasoning_content': 'The user says: "Hi, My name is Dev and I am a Software Engineer". Probably they are greeting. The assistant should respond politely, maybe ask how can help. There\'s no disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 83, 'total_tokens': 183, 'completion_time': 0.211236792, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.003560484, 'prompt_tokens_details': None, 'queue_time': 0.384090792, 'total_time': 0.214797276}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d997-c128-7c01-959b-b

### Maintaining Context Manually

In [91]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Dev and I am a Software Engineer"),
        AIMessage(content="Hi Dev! 👋 Great to meet you. How can I help you today? Whether you’ve got a coding question, need some design advice, or just want to chat about the latest tech trends, I’m here. 🚀"),
        HumanMessage(content="Hey, What's my name and What do I do?")
    ]
)

AIMessage(content='Your name is **Dev**, and you work as a **Software Engineer**. 🚀', additional_kwargs={'reasoning_content': 'The user asks: "Hey, What\'s my name and What do I do?" We have prior conversation: user said "Hi, My name is Dev and I am a Software Engineer". So we answer: name is Dev, you are a Software Engineer. Should respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 151, 'total_tokens': 234, 'completion_time': 0.172278083, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.006754447, 'prompt_tokens_details': None, 'queue_time': 0.342105591, 'total_time': 0.17903253}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d997-c49e-7cd0-9bd6-53b717b37f45-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 83, 'total_tokens'

## Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

### Importing Message History Components

In [92]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

### Creating Session-Based Message History

In [93]:
store = {}

def get_session_history(session_id : str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

### Creating a Stateful Chatbot

In [94]:
with_message_history = RunnableWithMessageHistory(model, get_session_history)

### Using Session IDs

In [95]:
config = {"configurable":{"session_id":"chat1"}}

In [96]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Dev and I am a Software Engineer")
    ],
    config = config
)
response.content

'Hello Dev! 👋 Great to meet a fellow software engineer. How can I assist you today? Whether it’s a coding question, architectural advice, career tips, or anything else, I’m here to help.'

In [97]:
with_message_history.invoke(
    [
        HumanMessage(content="What's my name?")
    ],
    config = config
)

AIMessage(content='Your name is Dev.', additional_kwargs={'reasoning_content': 'The user asks "What\'s my name?" The conversation: user introduced themselves as "Dev". So answer: "Your name is Dev." Should be concise.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 140, 'total_tokens': 186, 'completion_time': 0.097983105, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.005569733, 'prompt_tokens_details': None, 'queue_time': 0.214214695, 'total_time': 0.103552838}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d997-ca98-76b2-9d05-ecfd593247e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 140, 'output_tokens': 46, 'total_tokens': 186, 'output_token_details': {'reasoning': 32}})

### Switching Between Conversations

In [98]:
### Change the config (means changing the session_id)

config1 = {"configurable":{"session_id":"chat2"}}

In [99]:
response = with_message_history.invoke(
    [
        HumanMessage(content="What's my name?")
    ],
        config = config1
)
response.content

'I don’t have any information about your name. If you’d like, you can tell me what you’d like to be called, and I’ll use that name in our conversation.'

In [100]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hey, My name is IronMan")
    ],
        config = config1
)
response.content

'Nice to meet you, IronMan! How can I assist you today?'

In [101]:
response = with_message_history.invoke(
    [
        HumanMessage(content="What's my name?")
    ],
        config = config1
)
response.content

'Your name is IronMan.'

## Prompt Templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

### Creating a Chat Prompt Template

In [102]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [103]:
chain.invoke({"messages":[HumanMessage(content="Hi, My name is Dev")]})

AIMessage(content='Hello Dev! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as helpful assistant. The user says "Hi, My name is Dev". We can greet and ask how to help.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 97, 'total_tokens': 150, 'completion_time': 0.110487615, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.003682702, 'prompt_tokens_details': None, 'queue_time': 0.305210053, 'total_time': 0.114170317}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d997-d2ff-74c3-b4c1-64adc04b3e78-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 53, 'total_tokens': 150, 'output_token_details': {'reasoning': 29}})

In [104]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [105]:
config = {"configurable":{"session_id":"chat3"}}

In [106]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Dev")
    ],
    config=config
)
response.content

'Hello Dev! Nice to meet you. How can I help you today?'

### Using MessagesPlaceholder

In [107]:
## Add more complexity

prompt=ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt|model

## Adding Multiple Input Variables

### Adding Language as an Input Variable

In [108]:
response = chain.invoke({
    "messages":[HumanMessage(content="Hi, My name is Dev")],
    "language":"Hindi"
})

response.content

'नमस्ते देव! आपसे मिलकर खुशी हुई। मैं आपकी कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

### Integrating Prompt Templates with Message History

In [109]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [110]:
config = {"configurable":{"session_id":"chat4"}}

In [111]:
response = with_message_history.invoke(
    {"messages" : [HumanMessage(content="Hi, I am Dev Goyal")],
    "language": "Hindi"
    },
    config=config
)
response.content

'नमस्ते देव गोयल! आपसे मिलकर ख़ुशी हुई। मैं आपकी किस प्रकार मदद कर सकता हूँ?'

In [112]:
response = with_message_history.invoke(
    {"messages" : [HumanMessage(content="What's my name?")],
    "language": "Hindi"
    },
    config=config
)
response.content

'आपका नाम **देव गोयल** है।'

## Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

### Limiting Conversation History with trim_messages

In [113]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",              # focus on last conversation
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [114]:
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

### Building a Chain with Message Trimming

In [115]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)

response.content

'Based on what you mentioned earlier, you like vanilla ice cream! 🍦'

### Testing the Trimmed Conversation

In [116]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What math problem did i ask?")],
        "language": "English",
    }
)
response.content

'You asked, “What’s\u202f2\u202f+\u202f2?”'

### Integrating Message Trimming with Message History

In [117]:
## Lets wrap this in the Message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [118]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'Your name is Bob.'

In [119]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You asked about the addition problem “What’s\u202f2\u202f+\u202f2?” (which equals\u202f4).'